# Chroma VectorStore

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 벡터 저장소(VectorStore)의 역할과 Chroma의 특징을 설명할 수 있어요
- 문서를 Chroma에 저장하고 유사도 검색을 수행할 수 있어요
- 메타데이터 필터링, MMR 검색, Retriever 변환을 활용할 수 있어요
- RAG 체인에 Chroma를 통합할 수 있어요

## 사전 지식

- 6-3 Embeddings 학습 완료 (임베딩 개념 이해 필수)
- OpenAI API 키 설정

---

## VectorStore란?

벡터 저장소(VectorStore)는 임베딩 벡터를 저장하고 유사도 기반 검색을 제공하는 데이터베이스예요. RAG 시스템에서 "문서 창고" 역할을 해요.

```mermaid
flowchart LR
    subgraph 저장["저장 단계 (색인)"]
        D["문서들"]:::input
        E["임베딩 모델"]:::process
        VS["VectorStore<br/>(Chroma)"]:::storage
        D --> E --> VS
    end

    subgraph 검색["검색 단계 (쿼리)"]
        Q["사용자 질문"]:::input
        QE["임베딩 모델"]:::process
        QV["질문 벡터"]:::process
        R["유사 문서<br/>Top-K"]:::output
        Q --> QE --> QV
        QV -->|"유사도 비교"| VS
        VS --> R
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## Chroma 특징

- 로컬 개발에 최적화된 오픈소스 벡터 DB
- `persist_directory`로 디스크에 영구 저장
- 메타데이터 필터링으로 정밀 검색
- LangChain과 원활하게 통합

**참고 문서**: [Chroma 공식 문서](https://docs.trychroma.com/) | [LangChain Chroma 통합](https://python.langchain.com/docs/integrations/vectorstores/chroma/)

## 환경 설정

In [ ]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


---

## 1. 데이터 준비

벡터 저장소에 넣을 문서를 로드하고 분할해볼게요. NLP 키워드 문서와 금융 키워드 문서를 사용해요.

In [ ]:

# ---------------------------------------------------
# 문서 로드 및 분할 (벡터 저장소에 넣을 데이터 준비)
# ---------------------------------------------------

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할 설정 (chunk_size: 600자, overlap: 없음)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)

# 1단계: 문서 로더 생성
loader_nlp = TextLoader("data/nlp-keywords.txt")
loader_finance = TextLoader("data/finance-keywords.txt")

# 2단계: 문서 로드 및 분할
# load_and_split(): 로드 + 분할을 한 번에 처리하는 편의 메서드
docs_nlp = loader_nlp.load_and_split(text_splitter)
docs_finance = loader_finance.load_and_split(text_splitter)

# 3단계: 문서 개수 확인
print(f"NLP 문서 개수: {len(docs_nlp)}")
print(f"금융 문서 개수: {len(docs_finance)}")

# 문서 샘플 확인
print("\n[NLP 문서 샘플]")
print(docs_nlp[0].page_content[:200])


---

## 2. VectorStore 생성하기

### 2.1 from_documents로 생성

`from_documents()`는 Document 객체 리스트를 받아서 임베딩 후 저장소를 만들어주는 가장 편리한 방법이에요.

**주요 파라미터**:
- `documents`: Document 객체 리스트
- `embedding`: 임베딩 모델 (`OpenAIEmbeddings` 등)
- `collection_name`: 컬렉션 이름 (네임스페이스 역할)
- `persist_directory`: 디스크 저장 경로 (지정 안 하면 메모리에만 저장)

In [ ]:
# ---------------------------------------------------
# Chroma VectorStore 생성 (메모리)
# ---------------------------------------------------

# ============================================================
# TODO: from_documents()로 Chroma 벡터 저장소를 생성해보세요
# 힌트: Chroma.from_documents(documents, embedding, collection_name=...)
# 예상 결과: "VectorStore가 메모리에 생성되었습니다." 출력
# ============================================================

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1단계: Chroma.from_documents()로 메모리 기반 VectorStore 생성
# Chroma: 로컬 개발에 최적화된 오픈소스 벡터 DB
# collection_name: 같은 DB 안에서 데이터를 구분하는 네임스페이스


print("✅ VectorStore가 메모리에 생성되었습니다.")

### 2.2 디스크에 영구 저장하기

`persist_directory`를 지정하면 프로세스를 재시작해도 데이터가 유지돼요.

> **실무 팁**: 운영 환경에서는 항상 `persist_directory`를 지정해서 저장해두세요. 임베딩 API 비용이 다시 발생하지 않아요.

In [ ]:
# ---------------------------------------------------
# Chroma VectorStore 생성 (디스크 영구 저장)
# ---------------------------------------------------

# ============================================================
# TODO: persist_directory를 지정해 디스크에 저장되는 VectorStore를 만들어보세요
# 힌트: Chroma.from_documents(..., persist_directory="./chroma_tech_db")
# 예상 결과: 지정 경로에 폴더가 생성되고 "저장되었습니다." 출력
# ============================================================

# 1단계: 디스크 저장 경로 지정
DB_PATH = "./chroma_tech_db"

# 2단계: persist_directory로 디스크에 영구 저장
# persist_directory: 지정 시 프로세스 재시작 후에도 데이터 유지


print(f"✅ VectorStore가 '{DB_PATH}' 경로에 저장되었습니다.")

### 2.3 저장된 VectorStore 로드하기

이미 저장된 벡터 저장소를 다시 불러올 수 있어요. 임베딩을 새로 계산하지 않아도 돼요.

In [ ]:

# ---------------------------------------------------
# 디스크에 저장된 Chroma VectorStore 로드
# ---------------------------------------------------

# 1단계: 저장된 VectorStore 로드 (임베딩 재계산 불필요)
# Chroma(): from_documents() 대신 로드 전용 생성자 사용
loaded_vectorstore = Chroma(
    persist_directory=DB_PATH,
    embedding_function=OpenAIEmbeddings(),
    collection_name="tech_keywords"
)

# 2단계: 저장된 데이터 개수 확인
stored_data = loaded_vectorstore.get()
print(f"저장된 문서 개수: {len(stored_data['ids'])}")


---

## 3. 유사도 검색 (Similarity Search)

### 3.1 기본 유사도 검색

`similarity_search()`는 쿼리와 가장 유사한 문서 k개를 반환해요. 내부적으로 코사인 유사도를 계산해서 순위를 매겨요.

In [ ]:
# ---------------------------------------------------
# 기본 유사도 검색 (similarity_search)
# ---------------------------------------------------

# ============================================================
# TODO: similarity_search()로 쿼리와 유사한 문서를 검색해보세요
# 힌트: loaded_vectorstore.similarity_search("쿼리 텍스트")
# 예상 결과: 기본 4개 문서가 유사도 순으로 반환
# ============================================================

# 1단계: 기본 검색 실행 (k=4, 기본값)
# similarity_search(): 쿼리를 자동으로 임베딩 후 유사 문서 k개 반환


print("=" * 80)
print("검색 결과 (기본 4개)")
print("=" * 80)
for idx, doc in enumerate(results, 1):
    print(f"\n[{idx}] {doc.metadata.get('source', 'Unknown')}")
    print(doc.page_content[:150])

### 3.2 검색 결과 개수 조정

`k` 파라미터로 반환받을 문서 개수를 조절해요.

In [ ]:

# ---------------------------------------------------
# k 파라미터로 검색 결과 개수 조정
# ---------------------------------------------------

# k=2: 상위 2개 문서만 반환
results_top2 = loaded_vectorstore.similarity_search("임베딩과 벡터 검색에 대해 알려줘", k=2)

print(f"검색 결과: {len(results_top2)}개")
for idx, doc in enumerate(results_top2, 1):
    print(f"\n[{idx}] {doc.page_content[:100]}...")


### 3.3 메타데이터 필터링

`filter` 파라미터로 메타데이터 조건에 맞는 문서만 골라서 검색할 수 있어요. 특정 출처나 카테고리의 문서만 검색할 때 유용해요.

> **실무 팁**: 여러 종류의 문서를 하나의 VectorStore에 저장할 때는 `source`, `category`, `date` 같은 메타데이터를 잘 설계해두면 나중에 원하는 범위만 검색할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 메타데이터 필터로 특정 출처 문서만 검색
# ---------------------------------------------------

# ============================================================
# TODO: filter 파라미터로 특정 source의 문서만 검색해보세요
# 힌트: similarity_search(..., filter={"source": "경로"})
# 예상 결과: nlp-keywords.txt 출처 문서만 반환
# ============================================================

# 1단계: filter로 메타데이터 조건 지정
# filter: Chroma에서 메타데이터 조건으로 검색 범위를 좁힘


print("[필터링된 검색 결과]")
for idx, doc in enumerate(filtered_results, 1):
    print(f"\n{idx}. Source: {doc.metadata['source']}")
    print(f"   {doc.page_content[:100]}...")

---

## 4. 문서 추가 및 관리

### 4.1 문서 추가

기존 VectorStore에 새 문서를 추가할 수 있어요. 전체를 다시 만들 필요 없어요.

In [ ]:
# ---------------------------------------------------
# 기존 VectorStore에 새 문서 추가
# ---------------------------------------------------

from langchain_core.documents import Document

# 1단계: 새 문서 추가
# add_documents(): 기존 저장소를 유지하면서 새 문서를 덧붙임
new_doc_ids = loaded_vectorstore.add_documents(
    documents=[
        Document(
            page_content="Multi-Head Attention은 트랜스포머의 핵심 메커니즘입니다.",
            metadata={"source": "custom_data", "topic": "attention"}
        )
    ]
)

print(f"✅ {len(new_doc_ids)}개 문서가 추가되었습니다.")
print(f"추가된 ID: {new_doc_ids}")


---

## 5. Retriever로 변환

### 5.1 기본 Retriever

`as_retriever()`로 VectorStore를 Retriever(리트리버)로 변환하면 LangChain 체인에서 표준 인터페이스로 사용할 수 있어요.

In [ ]:
# ---------------------------------------------------
# VectorStore를 Retriever로 변환 (체인 통합용)
# ---------------------------------------------------

# ============================================================
# TODO: as_retriever()로 VectorStore를 Retriever로 변환해보세요
# 힌트: loaded_vectorstore.as_retriever()
# 예상 결과: 검색된 문서 4개 출력
# ============================================================

# 1단계: Retriever 생성
# as_retriever(): VectorStore를 LangChain 표준 검색 인터페이스로 변환
# Retriever는 invoke() 메서드를 지원하여 체인에 바로 연결 가능


# 2단계: 검색 실행


print(f"검색된 문서 개수: {len(retrieved_docs)}")
print("\n[검색 결과]")
for idx, doc in enumerate(retrieved_docs, 1):
    print(f"{idx}. {doc.page_content[:80]}...")

### 5.2 MMR (Maximum Marginal Relevance) 검색

MMR(최대 한계 관련성)은 유사도와 다양성을 동시에 고려하는 검색 방법이에요. 비슷한 문서가 중복으로 나오는 문제를 줄여줘요.

**주요 파라미터**:
- `k`: 최종 반환 문서 개수
- `fetch_k`: 후보 문서 개수 (이 중에서 k개 선택)
- `lambda_mult`: 0이면 최대 다양성, 1이면 최대 유사도

> **실무 팁**: 검색 결과가 너무 비슷한 문서들로만 채워진다면 MMR을 적용해보세요. `lambda_mult=0.5` 정도로 시작해서 조절해보세요.

In [ ]:
# ---------------------------------------------------
# MMR 검색으로 다양성을 고려한 Retriever 설정
# ---------------------------------------------------

# ============================================================
# TODO: search_type="mmr"로 MMR 방식 Retriever를 설정해보세요
# 힌트: as_retriever(search_type="mmr", search_kwargs={"k": ..., "fetch_k": ..., "lambda_mult": ...})
# 예상 결과: 유사도만 고려했을 때보다 다양한 주제의 문서 반환
# ============================================================

# 1단계: MMR Retriever 설정
# search_type="mmr": MMR(Maximum Marginal Relevance) 알고리즘 사용
# fetch_k: 유사도로 뽑을 후보 문서 수 (이 중에서 k개 선택)
# lambda_mult: 0 = 최대 다양성, 1 = 최대 유사도 (0.5가 균형)


mmr_results = mmr_retriever.invoke("자연어 처리와 임베딩 기술")

print("=== MMR 검색 결과 (다양성 고려) ===")
for idx, doc in enumerate(mmr_results, 1):
    print(f"\n[{idx}] {doc.page_content[:120]}...")

---

## 6. RAG 체인 구성

Retriever를 활용해서 간단한 RAG(검색 증강 생성) 체인을 만들어볼게요.

In [ ]:
# ---------------------------------------------------
# Retriever를 활용한 RAG 체인 구성 및 실행
# ---------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1단계: 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """다음 문서를 참고하여 질문에 답변해주세요.

참고 문서:
{context}

질문: {question}

답변:"""
)

# 2단계: 문서 포맷 함수
def format_docs(docs):
    """검색된 문서를 하나의 문자열로 결합"""
    return "\n\n".join(doc.page_content for doc in docs)

# 3단계: RAG 체인 구성
# RunnablePassthrough(): 입력(question)을 그대로 통과시키는 패스스루 컴포넌트
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

# 4단계: 질문 실행
question = "트랜스포머와 어텐션 메커니즘에 대해 설명해주세요."
answer = rag_chain.invoke(question)

print("=" * 80)
print("질문:", question)
print("=" * 80)
print("\n답변:")
print(answer)


---

## 마무리

### 핵심 요약

Chroma VectorStore의 주요 기능을 정리해볼게요:

| 기능 | 메서드 | 설명 |
|------|--------|------|
| 저장소 생성 | `from_documents()` | Document 리스트로 생성 |
| 영구 저장 | `persist_directory=` | 디스크에 저장 |
| 기본 검색 | `similarity_search()` | 유사도 기반 Top-K |
| 다양성 검색 | `as_retriever(search_type="mmr")` | MMR 알고리즘 |
| 필터 검색 | `filter={"key": "val"}` | 메타데이터 조건 |
| 문서 추가 | `add_documents()` | 기존 저장소에 추가 |
| Retriever | `as_retriever()` | 체인 통합 |

**Chroma는 이럴 때 사용해요**: 로컬 개발, 프로토타입 제작, 수천~수만 개 문서 처리, 재시작 후 데이터 유지가 필요한 소규모 프로젝트

### 다음 학습

**6-4 노트북 02**: FAISS — 대규모 데이터 고속 검색에 최적화된 벡터 저장소를 배워볼게요.